# Market Language Model: baseline training

Pipeline: load candles -> candle-geometry features with trailing volatility normalization -> triple-barrier labels with timeout and conservative tie-break -> non-overlapping sampling with strict temporal splits -> LightGBM baseline -> cost-adjusted expectancy.

Inputs deliberately exclude indicators, absolute prices, and coin identity. The cross-asset test at the end checks whether any edge generalizes or is asset-specific. AUC above 0.5 that holds on a coin the model never trained on is the core evidence that the normalized geometry carries transferable structure.

In [ ]:
# Colab/Kaggle setup: clone the repo if missing, then cd into it. Safe to
# re-run after a kernel restart (will not re-clone an existing folder).
import importlib.util, os, subprocess, sys
if importlib.util.find_spec('mlm') is None:
    if not os.path.isdir('CG-ALGO') and not os.path.isdir('mlm'):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/iceyxsm/CG-ALGO'], check=True)
    if os.path.isdir('CG-ALGO'):
        os.chdir('CG-ALGO')
if importlib.util.find_spec('lightgbm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'lightgbm'], check=True)
print('cwd', os.getcwd())

In [ ]:
import mlm
from mlm import (FeatureConfig, BarrierConfig,
                 SplitConfig, transfer_matrix)
from mlm.metrics import CostConfig

feat_cfg = FeatureConfig(vol_window=100)
bar_cfg = BarrierConfig(tp=0.02, sl=0.01, horizon=48)
split_cfg = SplitConfig(window=64, non_overlap=True)
cost_cfg = CostConfig(fee=0.0005, slippage=0.0005)

In [ ]:
# Per-asset OHLCV: fetch real Binance 5m history, cached to CSV so a kernel
# restart does not re-download. Add or remove symbols as needed.
from mlm import load_ohlcv_csv
from fetch_data import fetch_to_csv
SYMBOLS = {'BTC': 'BTCUSDT', 'ETH': 'ETHUSDT', 'DOGE': 'DOGEUSDT'}
assets = {k: load_ohlcv_csv(fetch_to_csv(v, '5m', days=3300))
          for k, v in SYMBOLS.items()}
{k: len(v) for k, v in assets.items()}

In [ ]:
# Transfer matrix. Diagonal cells (train == test) are the within-asset ceiling;
# off-diagonal cells are the cross-asset test. The gap is the generalization
# tax. Pooled rows (e.g. BTC+ETH -> DOGE) stack per-asset datasets, never raw
# series. Thresholds are calibrated on each cell's validation split only.
cells = [
    (['BTC'], 'BTC'), (['ETH'], 'ETH'), (['DOGE'], 'DOGE'),   # ceilings
    (['BTC'], 'ETH'), (['ETH'], 'BTC'), (['BTC'], 'DOGE'),    # cross-asset
    (['BTC', 'ETH'], 'DOGE'),                                 # pooled transfer
]
# Keep only cells whose assets were actually loaded, so a BTC-only fetch runs
# just the BTC->BTC ceiling instead of raising a KeyError on ETH/DOGE.
cells = [(tr, te) for tr, te in cells
         if all(a in assets for a in tr) and te in assets]
results = transfer_matrix(assets, cells, feat_cfg, bar_cfg, split_cfg, cost_cfg)
import pandas as pd
pd.DataFrame(results)[['train', 'test', 'within_asset', 'auc', 'log_loss',
                       'base_log_loss', 'n_test', 'n_taken', 'expectancy',
                       'exp_ci_lo', 'exp_ci_hi']]

Read each row's `expectancy` as mean return per taken trade after round-trip costs, with trades taken only above the calibrated breakeven threshold. Compare each cross-asset row against its within-asset ceiling (the diagonal for the test asset): a small gap means the representation transfers, a large gap means the model learned an asset dialect. Watch `n_taken` and the bootstrap interval `[exp_ci_lo, exp_ci_hi]` together -- a positive expectancy whose lower bound is still below zero is not distinguishable from noise, no matter how good the point estimate looks. On synthetic random-walk data every cell should hover near zero with intervals straddling it, which confirms the framework is honest rather than leaking.

## CNN rung

Same transfer matrix, but a 1D-CNN replaces LightGBM. The tree flattened the window and had to relearn a pattern at every position; the CNN slides filters across time, so a motif is detected wherever it occurs. If the CNN's within-asset ceiling clears the cost hurdle where the tree did not, locality carried the signal. If it also lands at zero, that is stronger evidence the geometry alone lacks edge. Needs a GPU runtime to be quick.

In [ ]:
from mlm.cnn import train_cnn, predict_cnn
cnn_results = transfer_matrix(assets, cells, feat_cfg, bar_cfg, split_cfg,
                              cost_cfg, fit=train_cnn, predict=predict_cnn)
pd.DataFrame(cnn_results)[['train', 'test', 'within_asset', 'dir_auc',
                           'n_resolved', 'auc', 'n_test']]

## Transformer rung

Each candle is a token with positional encoding; self-attention reads long-range structure across the whole window. This is the model built for the grammar hypothesis. Read `dir_auc` (WIN vs LOSS, TIMEOUT dropped): if it rises meaningfully above the tree and CNN with the same cross-asset transfer, sequential long-range structure carries signal the simpler models could not see. If it matches them, the signal is essentially local. Use a GPU runtime.

In [ ]:
from mlm.transformer import train_transformer, predict_transformer
tf_results = transfer_matrix(assets, cells, feat_cfg, bar_cfg, split_cfg,
                             cost_cfg, fit=train_transformer,
                             predict=predict_transformer)
pd.DataFrame(tf_results)[['train', 'test', 'within_asset', 'dir_auc',
                          'n_resolved', 'auc', 'n_test']]

## Masked-candle pretraining (language-model rung)

Pretrain the encoder by masking candle tokens and reconstructing them, on dense overlapping windows from the training asset's train-region only (the test split and any target asset stay unseen), then fine-tune for direction. This uses far more data than the sparse labels. If `dir_auc` beats the from-scratch transformer's ~0.59, representation learning unlocked signal the labels alone could not; if it matches, the directional signal is local and fully captured, closing stage one.

In [ ]:
from mlm.masked import dense_windows, train_masked, predict_masked
from mlm import build_splits, skill, WIN, LOSS
import numpy as np

# Train on BTC with masked pretraining, evaluate direction on each target.
src = 'BTC'
s_src = build_splits(assets[src], feat_cfg, bar_cfg, split_cfg)
pre_X = dense_windows(assets[src], window=split_cfg.window, end_frac=0.7)
model = train_masked(s_src['train'], s_src['val'], pretrain_X=pre_X)

rows = []
for tgt in assets:
    Xte, yte, _ = build_splits(assets[tgt], feat_cfg, bar_cfg, split_cfg)['test']
    p = predict_masked(model, Xte)
    m = (yte == WIN) | (yte == LOSS)
    rows.append({'train': src, 'test': tgt,
                 'dir_auc': round(skill(yte[m], p[m])['dir_auc'], 4),
                 'n_resolved': int(m.sum())})
pd.DataFrame(rows)